## PropertyLens RAG — Notebook A: Build & Upsert (v4)

**Purpose:** ingestion pipeline that loads raw HDB data, builds chunks, encodes them, and upserts to Pinecone. Run once per data refresh.

**Why v4 instead of v3:**
- V3 crashed the kernel in the smoke-test cell after loading 1.57M transaction chunks + 204k CBR rows + BGE-M3 + cross-encoder while Ollama was serving Gemma. Classic OOM.
- V4 fix: apply `SAMPLE_SIZE` *before* chunk building, cap CBR at read time, encode/upsert one batch at a time with the batch going out of scope each iteration, and drop chunk lists the moment their namespace is upserted.

**Output:** populated Pinecone index + `bm25_encoder_v3.pkl` on disk. Notebook B (`05_propertylens_rag_inference.ipynb`) reads both.

![PropertyLens RAG build & upsert pipeline](../../images/Screenshot%202026-04-17%20at%201.29.02%E2%80%AFPM.png)


### Install dependencies

In [1]:
# Build-time dependencies. psutil is new — used for memory checkpoints.
!pip install -q pinecone pinecone-text sentence-transformers transformers torch \
               pyarrow pandas numpy tqdm python-dotenv psutil

### Configuration

Single biggest lever here is `SAMPLE_SIZE`. V3 loaded 1.57M transactions then threw 1.569M of them away during sampling — V4 slices first so every downstream stage sees at most `SAMPLE_SIZE` rows.


In [2]:
from __future__ import annotations
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── Pinecone ──────────────────────────────────────────────────────────────────
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX   = "propertylens-rag"
NS_TRANSACTIONS  = "transactions"
NS_AMENITIES     = "amenities"
NS_XAI           = "xai"
NS_TRENDS        = "trends"

assert PINECONE_API_KEY, "Missing PINECONE_API_KEY in repo-root .env"

# ── Models ────────────────────────────────────────────────────────────────────
DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DIMENSION  = 1024

# ── Ingestion knobs ───────────────────────────────────────────────────────────
# None = full corpus (slow, ~1.57M rows). 1000 = fast sample for dev.
SAMPLE_SIZE: int | None = 1000

# BM25 fit corpus cap. Only used on cache miss.
BM25_MAX_TRANSACTION_TEXTS: int | None = 400_000

# Cap CBR rows at read time. V3 held 204k rows to use 200.
CBR_MAX_ROWS = 500

# Smaller = safer on memory, slightly slower.
UPSERT_BATCH_SIZE = 64

# ── Paths ─────────────────────────────────────────────────────────────────────
def find_repo_root(start: Path | None = None) -> Path:
    """Walk up until we find a dir with both data/ and notebooks/ children."""
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Cannot find repo root — expected data/ and notebooks/ siblings.")

REPO_ROOT         = find_repo_root()
TRANSACTIONS_GLOB = str(REPO_ROOT / "data" / "feature_data" / "**" / "outputs" / "*.csv")
AMENITIES_GLOB    = str(REPO_ROOT / "data" / "amenities" / "*.csv")
XAI_DIR           = str(REPO_ROOT / "data" / "artifacts" / "hybrid_xai")

# BM25 cache — shared with notebook B. Do NOT change without updating B.
BM25_CACHE_PATH = REPO_ROOT / "notebooks" / "05_chatbot" / "bm25_encoder_v3.pkl"

print("Config loaded.")
print(f"  Pinecone index    : {PINECONE_INDEX}")
print(f"  REPO_ROOT         : {REPO_ROOT}")
print(f"  SAMPLE_SIZE       : {SAMPLE_SIZE}")
print(f"  BM25 cache path   : {BM25_CACHE_PATH}")


Config loaded.
  Pinecone index    : propertylens-rag
  REPO_ROOT         : /Users/bhuvesh/Documents/PropertyLens
  SAMPLE_SIZE       : 1000
  BM25 cache path   : /Users/bhuvesh/Documents/PropertyLens/notebooks/05_chatbot/bm25_encoder_v3.pkl


### Memory helpers

`mem(label)` is the single most useful debugging tool in this notebook. It forces garbage collection and prints resident set size, so you can watch RAM grow and shrink stage-by-stage. If V3 had this, we'd have found the OOM in 5 minutes.


In [3]:
from __future__ import annotations
import gc
import psutil

_PROC = psutil.Process(os.getpid())

def rss_mb() -> float:
    """Return current resident set size in MB."""
    return _PROC.memory_info().rss / (1024 * 1024)

def mem(label: str) -> None:
    """Print a memory checkpoint and force garbage collection."""
    gc.collect()
    print(f"  [MEM] {label:<32s} RSS = {rss_mb():8.1f} MB")

mem("startup")


  [MEM] startup                          RSS =     71.6 MB


### Load raw data

Two memory-conscious changes vs V3:

1. **Apply `SAMPLE_SIZE` immediately** — `load_transactions` slices before returning, so downstream code never sees the full corpus.
2. **Cap CBR rows at read time** — after reading the parquet we immediately take `.head(CBR_MAX_ROWS).copy()` and `del` the full DataFrame. Yes, parquet read still briefly materialises the whole file, but the ~203.5k-row surplus is released before any heavy model is loaded.


In [4]:
from __future__ import annotations
import glob
import json
import pandas as pd


def _infer_from_onehots(df: pd.DataFrame, prefix: str) -> pd.Series | None:
    """Infer a categorical value from one-hot columns like town_* or flat_type_*."""
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        return None
    return df[cols].idxmax(axis=1).str.replace(prefix, "", regex=False)


def load_transactions(pattern: str, sample_size: int | None) -> pd.DataFrame:
    """Load transaction CSVs. If sample_size is set, slice immediately."""
    files = glob.glob(pattern, recursive=True)
    if not files:
        raise FileNotFoundError(f"No CSVs found: {pattern}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

    if "town" not in df.columns:
        town = _infer_from_onehots(df, "town_")
        if town is not None:
            df["town"] = town
    if "flat_type" not in df.columns:
        flat_type = _infer_from_onehots(df, "flat_type_")
        if flat_type is not None:
            df["flat_type"] = flat_type
    if "town" in df.columns:
        df["town"] = df["town"].astype(str).str.upper().str.strip()
    if "flat_type" in df.columns:
        df["flat_type"] = df["flat_type"].astype(str).str.upper().str.strip()
    if "transaction_year" not in df.columns and "month" in df.columns:
        df["transaction_year"] = pd.to_datetime(df["month"], errors="coerce").dt.year

    total = len(df)
    # Apply sample early — biggest memory win vs V3
    if sample_size is not None and total > sample_size:
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
        print(f"  Transactions : sampled {sample_size:,} of {total:,} rows from {len(files)} file(s)")
    else:
        print(f"  Transactions : {total:,} rows from {len(files)} file(s) (full corpus)")
    return df


def load_amenities(pattern: str) -> pd.DataFrame:
    """Load all amenity CSVs, tagging each row with its source filename."""
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"No CSVs found: {pattern}")
    dfs = []
    for f in files:
        tmp = pd.read_csv(f)
        tmp["source_file"] = os.path.basename(f)
        dfs.append(tmp)
    df = pd.concat(dfs, ignore_index=True)
    print(f"  Amenities    : {len(df):,} rows from {len(files)} file(s)")
    return df


def derive_trends(transactions: pd.DataFrame) -> pd.DataFrame:
    """Median resale_price per town per year. Operates on sampled df."""
    if "town" not in transactions.columns:
        raise KeyError("transactions has no 'town' column")
    year_col = "transaction_year" if "transaction_year" in transactions.columns else "year"
    grp = (
        transactions.groupby(["town", year_col])["resale_price"]
        .agg(median_resale_price="median", transaction_count="count")
        .reset_index()
        .rename(columns={year_col: "year"})
    )
    print(f"  Trends       : {len(grp):,} town-year rows")
    return grp


def load_xai_bundle(xai_dir: str, cbr_max_rows: int) -> dict:
    """Load SHAP + rules + CBR. CBR is capped early."""
    bundle: dict = {}
    if not os.path.isdir(xai_dir):
        print(f"  WARNING: XAI dir not found: {xai_dir}")
        return bundle

    shap_path = os.path.join(xai_dir, "global_shap_cache.json")
    if os.path.exists(shap_path):
        with open(shap_path) as f:
            bundle["global_shap"] = json.load(f)
        print("  XAI SHAP     : loaded global_shap_cache.json")

    rules_path = os.path.join(xai_dir, "rules.json")
    if os.path.exists(rules_path):
        with open(rules_path) as f:
            bundle["rules"] = json.load(f)
        print("  XAI rules    : loaded rules.json")

    cbr_path = os.path.join(xai_dir, "cbr_training_data.parquet")
    if os.path.exists(cbr_path):
        full = pd.read_parquet(cbr_path)
        bundle["cbr_data"] = full.head(cbr_max_rows).copy()
        print(f"  XAI CBR      : kept {len(bundle['cbr_data']):,} / {len(full):,} rows "
              f"(capped at {cbr_max_rows})")
        del full  # release the ~203.5k-row surplus immediately

    if not bundle:
        print(f"  WARNING: no XAI artefacts found in {xai_dir}")
    return bundle


print("Loading data sources...")
transactions_df = load_transactions(TRANSACTIONS_GLOB, SAMPLE_SIZE)
amenities_df    = load_amenities(AMENITIES_GLOB)
trends_df       = derive_trends(transactions_df)
xai_bundle      = load_xai_bundle(XAI_DIR, CBR_MAX_ROWS)
mem("after loading raw data")


Loading data sources...
  Transactions : sampled 1,000 of 1,568,804 rows from 9 file(s)
  Amenities    : 615 rows from 4 file(s)
  Trends       : 257 town-year rows
  XAI SHAP     : loaded global_shap_cache.json
  XAI rules    : loaded rules.json
  XAI CBR      : kept 500 / 204,066 rows (capped at 500)
  [MEM] after loading raw data           RSS =   1775.3 MB


### Build transaction chunks

Child chunk (short, ~256 tokens) is sent to Pinecone as the `text`. Parent chunk (child + town amenity summary) is stored in metadata for cross-encoder scoring and LLM context. Logic identical to V3 — only difference is that `transactions_df` is already sampled.


In [5]:
from __future__ import annotations
import math, re
from typing import Any
import numpy as np


def _safe_float(x: Any) -> float | None:
    try:
        v = float(x)
        return v if np.isfinite(v) else None
    except Exception:
        return None


def _bucket_storey(storey_range: Any) -> str:
    if storey_range is None or (isinstance(storey_range, float) and math.isnan(storey_range)):
        return "unknown"
    m = re.search(r"(\d{1,2})\s*TO\s*(\d{1,2})", str(storey_range).upper())
    mid = (int(m.group(1)) + int(m.group(2))) / 2 if m else None
    if mid is None:
        m2 = re.search(r"(\d{1,2})", str(storey_range))
        mid = float(m2.group(1)) if m2 else None
    if mid is None:
        return "unknown"
    if mid <= 5:  return "01-05"
    if mid <= 12: return "06-12"
    if mid <= 20: return "13-20"
    return "21+"


def _bucket_price(price: float) -> str:
    if not np.isfinite(price) or price <= 0:
        return "unknown"
    lo = int(price // 100_000) * 100
    return f"{lo}k-{lo + 100}k"


HDB_TOWNS = (
    "ANG MO KIO", "BEDOK", "BISHAN", "BUKIT BATOK", "BUKIT MERAH",
    "BUKIT PANJANG", "BUKIT TIMAH", "CENTRAL AREA", "CHOA CHU KANG",
    "CLEMENTI", "GEYLANG", "HOUGANG", "JURONG EAST", "JURONG WEST",
    "KALLANG/WHAMPOA", "MARINE PARADE", "PASIR RIS", "PUNGGOL",
    "QUEENSTOWN", "SEMBAWANG", "SENGKANG", "SERANGOON", "TAMPINES",
    "TOA PAYOH", "WOODLANDS", "YISHUN",
)


def _infer_town(text: str) -> str:
    u = str(text).upper()
    for t in sorted(HDB_TOWNS, key=len, reverse=True):
        if t in u:
            return t
    return ""


def _ensure_town_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "town" in out.columns and out["town"].notna().any():
        out["town"] = out["town"].str.upper().str.strip()
        return out
    if "address" in out.columns:
        out["town"] = out["address"].map(_infer_town)
    else:
        out["town"] = ""
    return out


def build_town_amenity_summary(amenities: pd.DataFrame) -> dict[str, str]:
    amenities = _ensure_town_column(amenities)
    summary: dict[str, str] = {}
    for town, grp in amenities.groupby("town"):
        if not str(town).strip():
            continue
        lines: list[str] = []
        if "source_file" in grp.columns:
            for src, sg in grp.groupby("source_file"):
                atype  = str(src).replace(".csv", "").replace("_", " ").title()
                names  = sg["name"].dropna().astype(str).tolist()
                ex     = ", ".join(names[:3])
                suffix = ", ..." if len(names) > 3 else ""
                lines.append(f"{atype} ({len(names)}): {ex}{suffix}")
        else:
            names = grp["name"].dropna().astype(str).tolist()
            lines.append(f"Amenities ({len(names)}): {', '.join(names[:5])}")
        summary[str(town).upper().strip()] = "\n".join(lines)
    return summary


def _format_child(row: pd.Series) -> str:
    town      = str(row.get("town", "")).upper().strip()
    flat_type = str(row.get("flat_type", "")).upper()
    year      = int(row.get("transaction_year", 0) or 0)
    price     = _safe_float(row.get("resale_price")) or 0.0
    area      = _safe_float(row.get("floor_area_sqm"))
    psf       = (price / area / 10.7639) if area and area > 0 else None
    storey    = _bucket_storey(row.get("storey_range"))
    ctx       = f"This is an HDB resale transaction in {town}, year {year}."
    facts = [
        f"Town: {town}",
        f"Flat type: {flat_type}",
        f"Storey band: {storey}",
        f"Floor area (sqm): {area:.1f}" if area else "Floor area (sqm): unknown",
        f"Resale price (SGD): {int(round(price))}",
        f"Approx PSF (SGD): {psf:.1f}" if psf else "Approx PSF (SGD): unknown",
    ]
    return ctx + "\n" + "\n".join(facts)


def _format_parent(row: pd.Series, amenity_summary: str) -> str:
    base  = _format_child(row)
    lines = [base, "", "Nearby amenities (town-level):"]
    lines.append(amenity_summary if amenity_summary else "- No amenity data for this town.")
    return "\n".join(lines)


def build_transaction_chunks(
    transactions: pd.DataFrame,
    town_amenity_summary: dict[str, str],
) -> list[dict]:
    """Build Pinecone-ready chunk dicts. Operates on the already-sampled df."""
    out: list[dict] = []
    for i, row in transactions.reset_index(drop=True).iterrows():
        town      = str(row.get("town", "")).upper().strip()
        flat_type = str(row.get("flat_type", "")).upper()
        year      = int(row.get("transaction_year", 0) or 0)
        price     = _safe_float(row.get("resale_price")) or 0.0
        area      = _safe_float(row.get("floor_area_sqm"))
        psf       = (price / area / 10.7639) if area and area > 0 else None
        addr      = str(row.get("address_key", f"row_{i}")).upper().replace(" ", "_")
        out.append({
            "id":          f"txn_{addr}_{year}_{i}",
            "text":        _format_child(row),
            "parent_text": _format_parent(row, town_amenity_summary.get(town, "")),
            "metadata": {
                "source":       "transaction",
                "town":         town,
                "flat_type":    flat_type,
                "storey_band":  _bucket_storey(row.get("storey_range")),
                "sale_year":    year,
                "price_band":   _bucket_price(price),
                "resale_price": int(round(price)),
                "psf":          float(round(psf, 1)) if psf is not None else 0.0,
            },
        })
    return out


town_amenity_summary = build_town_amenity_summary(amenities_df)
txn_chunks = build_transaction_chunks(transactions_df, town_amenity_summary)
print(f"Transaction chunks : {len(txn_chunks):,}")
if txn_chunks:
    print("\nExample child:\n", txn_chunks[0]["text"])
mem("after txn chunks built")


Transaction chunks : 1,000

Example child:
 This is an HDB resale transaction in SERANGOON, year 2019.
Town: SERANGOON
Flat type: 5 ROOM
Storey band: unknown
Floor area (sqm): 144.0
Resale price (SGD): 770000
Approx PSF (SGD): 496.8
  [MEM] after txn chunks built           RSS =   1776.5 MB


### Build amenity chunks

In [6]:
from __future__ import annotations


def build_amenity_chunks(amenities: pd.DataFrame) -> list[dict]:
    amenities  = _ensure_town_column(amenities)
    group_cols = ["town", "source_file"] if "source_file" in amenities.columns else ["town"]
    out: list[dict] = []
    for keys, grp in amenities.groupby(group_cols):
        town = str(keys[0] if isinstance(keys, tuple) else keys).upper().strip()
        if not town:
            continue
        src   = str(keys[1] if isinstance(keys, tuple) and len(keys) > 1 else "amenities").replace(".csv", "")
        atype = src.replace("_", " ").title()
        names = grp["name"].dropna().astype(str).tolist()
        text  = (
            f"{atype} in {town} ({len(names)} total): {', '.join(names)}. "
            f"These are the {atype.lower()} amenities in the {town} HDB town."
        )
        out.append({
            "id":          f"amenity_{town}_{src}".replace(" ", "_").lower(),
            "text":        text,
            "parent_text": text,
            "metadata": {
                "source":       "amenity",
                "town":         town,
                "amenity_type": atype,
                "count":        len(names),
            },
        })
    return out


amenity_chunks = build_amenity_chunks(amenities_df)
print(f"Amenity chunks : {len(amenity_chunks):,}")
mem("after amenity chunks")


Amenity chunks : 85
  [MEM] after amenity chunks             RSS =   1776.6 MB


### Build trend chunks

In [7]:
from __future__ import annotations


def build_trend_chunks(trends: pd.DataFrame) -> list[dict]:
    out: list[dict] = []
    for _, row in trends.iterrows():
        town = str(row.get("town", "")).upper().strip()
        year = int(row.get("year", 0) or 0)
        med  = float(row.get("median_resale_price", 0) or 0)
        n    = int(row.get("transaction_count", 0) or 0)
        text = (
            f"HDB resale trend for {town}, year {year}: "
            f"median resale price SGD {int(round(med)):,} across {n:,} transactions."
        )
        out.append({
            "id":          f"trend_{town}_{year}",
            "text":        text,
            "parent_text": text,
            "metadata": {
                "source":            "trend",
                "town":              town,
                "sale_year":         year,
                "resale_price":      int(round(med)),
                "transaction_count": n,
            },
        })
    return out


trend_chunks = build_trend_chunks(trends_df)
print(f"Trend chunks : {len(trend_chunks):,}")
mem("after trend chunks")


Trend chunks : 257
  [MEM] after trend chunks               RSS =   1776.6 MB


### Build XAI chunks

In [8]:
from __future__ import annotations


def _shap_chunks(bundle: dict) -> list[dict]:
    out: list[dict] = []
    shap_data = bundle.get("global_shap")
    if shap_data is None:
        return out
    if isinstance(shap_data, dict):
        items = sorted(shap_data.items(), key=lambda x: abs(float(x[1])), reverse=True)[:20]
    elif isinstance(shap_data, list):
        items = shap_data[:20]
    else:
        return out
    lines = [f"  {feat}: {val:.4f}" for feat, val in items]
    text  = "Global SHAP feature importances for HDB price prediction:\n" + "\n".join(lines)
    out.append({
        "id": "xai_shap_global",
        "text": text,
        "parent_text": text,
        "metadata": {"source": "xai", "xai_type": "shap_global"},
    })
    return out


def _rule_chunks(bundle: dict) -> list[dict]:
    out: list[dict] = []
    rules = bundle.get("rules")
    if not rules:
        return out
    rule_list = rules if isinstance(rules, list) else list(rules.items())[:50]
    for i, rule in enumerate(rule_list[:50]):
        text = f"HDB pricing rule #{i+1}: {str(rule)}"
        out.append({
            "id": f"xai_rule_{i}",
            "text": text,
            "parent_text": text,
            "metadata": {"source": "xai", "xai_type": "rule", "rule_id": i},
        })
    return out


def _cbr_chunks(bundle: dict) -> list[dict]:
    out: list[dict] = []
    cbr = bundle.get("cbr_data")
    if cbr is None:
        return out
    rows = cbr.head(200).to_dict(orient="records") if isinstance(cbr, pd.DataFrame) else cbr[:200]
    for i, case in enumerate(rows):
        text = "CBR comparable case: " + ", ".join(f"{k}={v}" for k, v in list(case.items())[:12])
        out.append({
            "id": f"xai_cbr_{i}",
            "text": text,
            "parent_text": text,
            "metadata": {
                "source":   "xai",
                "xai_type": "cbr",
                "town":     str(case.get("town", "")).upper(),
            },
        })
    return out


def build_xai_chunks(bundle: dict) -> list[dict]:
    shap  = _shap_chunks(bundle)
    rules = _rule_chunks(bundle)
    cbr   = _cbr_chunks(bundle)
    print(f"  SHAP: {len(shap)} | Rules: {len(rules)} | CBR: {len(cbr)}")
    return shap + rules + cbr


xai_chunks = build_xai_chunks(xai_bundle)
print(f"XAI chunks total : {len(xai_chunks):,}")
mem("after xai chunks")


  SHAP: 1 | Rules: 3 | CBR: 200
XAI chunks total : 204
  [MEM] after xai chunks                 RSS =   1776.6 MB


### Release raw DataFrames

Chunks are built; the 1.57M-row transactions DataFrame, the amenities DataFrame, the trends DataFrame, and the XAI bundle (which still holds the CBR rows) are no longer needed. Drop them now before loading BGE-M3.


In [9]:
# Drop everything we no longer need
del transactions_df
del amenities_df
del trends_df
del xai_bundle
del town_amenity_summary

mem("after releasing raw data")


  [MEM] after releasing raw data         RSS =   1583.6 MB


### Initialise encoders

Loads BGE-M3 dense encoder. BM25 is either loaded from a pickle cache or fit from scratch (and cached for next time). Notebook B only loads from cache — it never refits.


In [10]:
from __future__ import annotations
import pickle
import random
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder


def _bm25_corpus_texts(
    txn_chunks: list,
    amenity_chunks: list,
    trend_chunks: list,
    xai_chunks: list,
    max_txn: int | None,
    seed: int = 42,
) -> list[str]:
    """Subsample transaction texts for BM25 fit if needed."""
    rng = random.Random(seed)
    txn = txn_chunks
    if max_txn is not None and len(txn_chunks) > max_txn:
        txn = rng.sample(txn_chunks, max_txn)
        print(f"  BM25 fit corpus: {len(txn):,} / {len(txn_chunks):,} transaction texts")
    merged = txn + amenity_chunks + trend_chunks + xai_chunks
    return [c["text"] for c in merged]


def fit_or_load_bm25(
    txn_chunks, amenity_chunks, trend_chunks, xai_chunks,
    max_txn: int | None, cache_path: Path,
) -> BM25Encoder:
    """Load BM25 from cache, or fit and cache."""
    if cache_path.exists():
        print(f"  BM25: loading from cache {cache_path}")
        with cache_path.open("rb") as f:
            return pickle.load(f)
    corpus_texts = _bm25_corpus_texts(
        txn_chunks, amenity_chunks, trend_chunks, xai_chunks, max_txn,
    )
    print(f"  BM25: fitting on {len(corpus_texts):,} documents...")
    enc = BM25Encoder()
    enc.fit(corpus_texts)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("wb") as f:
        pickle.dump(enc, f)
    print(f"  BM25: fitted and cached → {cache_path}")
    del corpus_texts
    return enc


dense_encoder = SentenceTransformer(DENSE_MODEL_NAME)
bm25_encoder  = fit_or_load_bm25(
    txn_chunks, amenity_chunks, trend_chunks, xai_chunks,
    BM25_MAX_TRANSACTION_TEXTS, BM25_CACHE_PATH,
)

print(f"Dense encoder : {DENSE_MODEL_NAME}")
print("BM25 encoder  : ready")
mem("after encoders loaded")


/Users/bhuvesh/Documents/PropertyLens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 42569.06it/s]


  BM25: loading from cache /Users/bhuvesh/Documents/PropertyLens/notebooks/05_chatbot/bm25_encoder_v3.pkl
Dense encoder : BAAI/bge-m3
BM25 encoder  : ready
  [MEM] after encoders loaded            RSS =   1210.3 MB


### Initialise Pinecone index

In [11]:
from __future__ import annotations
from pinecone import Pinecone, ServerlessSpec


def init_pinecone(api_key: str) -> Pinecone:
    return Pinecone(api_key=api_key)


def get_or_create_index(pc: Pinecone, index_name: str, dimension: int):
    existing = [idx.name for idx in pc.list_indexes()]
    if index_name not in existing:
        print(f"  Creating index '{index_name}'...")
        pc.create_index(
            name=index_name, dimension=dimension, metric="dotproduct",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
    else:
        print(f"  Index '{index_name}' already exists.")
    return pc.Index(index_name)


pc    = init_pinecone(PINECONE_API_KEY)
index = get_or_create_index(pc, PINECONE_INDEX, DENSE_DIMENSION)
print(index.describe_index_stats())


  Index 'propertylens-rag' already exists.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '278',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 05:10:06 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'transactions': {'vector_count': 995},
                'trends': {'vector_count': 312},
                'xai': {'vector_count': 204}},
 'storageFullness': 0.0,
 'total_vector_count': 159

### Encode and upsert — streaming, batch-by-batch

Encoding everything at once would materialise millions of dense vectors in RAM. Instead this loop:

1. Slices one batch of chunk dicts
2. Encodes only that batch
3. Upserts it to Pinecone
4. `del`s the batch and the records before the next iteration

After each namespace finishes, its chunk list is set to `[]` so we can't accidentally hold onto it. Peak memory ≈ one batch of vectors, not all of them.


In [12]:
from __future__ import annotations
from typing import Any
from tqdm import tqdm


def _encode_one(chunk: dict) -> dict:
    """Encode one chunk into a Pinecone upsert record."""
    dense  = dense_encoder.encode(chunk["text"], normalize_embeddings=True).tolist()
    sparse = bm25_encoder.encode_documents([chunk["text"]])[0]
    meta   = dict(chunk["metadata"])
    # Pinecone has a ~40KB metadata cap per vector; clamp parent_text
    meta["parent_text"] = chunk["parent_text"][:3000]
    return {"id": chunk["id"], "values": dense, "sparse_values": sparse, "metadata": meta}


def upsert_namespace(
    index, chunks: list[dict], namespace: str, batch_size: int,
) -> None:
    """
    Encode and upsert chunks batch-by-batch. Each batch's encoded records
    go out of scope at the end of the loop iteration, so peak memory is
    roughly one batch of vectors — not the full namespace.
    """
    n = len(chunks)
    if n == 0:
        print(f"  (no chunks for '{namespace}' — skipping)")
        return
    n_batches = (n + batch_size - 1) // batch_size
    for i in tqdm(range(0, n, batch_size),
                  desc=f"Encoding {namespace}",
                  total=n_batches):
        batch   = chunks[i : i + batch_size]
        records = [_encode_one(c) for c in batch]
        index.upsert(vectors=records, namespace=namespace)
        del records
        del batch
    print(f"  Upserted {n:,} → '{namespace}'")


# One namespace at a time. After each, DROP the chunk list.
upsert_namespace(index, txn_chunks, NS_TRANSACTIONS, UPSERT_BATCH_SIZE)
txn_chunks = []
mem("after transactions upsert")

upsert_namespace(index, amenity_chunks, NS_AMENITIES, UPSERT_BATCH_SIZE)
amenity_chunks = []
mem("after amenities upsert")

upsert_namespace(index, trend_chunks, NS_TRENDS, UPSERT_BATCH_SIZE)
trend_chunks = []
mem("after trends upsert")

upsert_namespace(index, xai_chunks, NS_XAI, UPSERT_BATCH_SIZE)
xai_chunks = []
mem("after xai upsert")

print("\nAll namespaces upserted.")
print(index.describe_index_stats())


Encoding transactions: 100%|██████████| 16/16 [00:36<00:00,  2.26s/it]


  Upserted 1,000 → 'transactions'
  [MEM] after transactions upsert        RSS =   1360.2 MB


Encoding amenities: 100%|██████████| 2/2 [00:05<00:00,  2.82s/it]


  Upserted 85 → 'amenities'
  [MEM] after amenities upsert           RSS =   1660.3 MB


Encoding trends: 100%|██████████| 5/5 [00:08<00:00,  1.69s/it]


  Upserted 257 → 'trends'
  [MEM] after trends upsert              RSS =   1708.0 MB


Encoding xai: 100%|██████████| 4/4 [00:09<00:00,  2.49s/it]


  Upserted 204 → 'xai'
  [MEM] after xai upsert                 RSS =   1809.0 MB

All namespaces upserted.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '279',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 05:11:06 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '3',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'transactions': {'vector_count': 1995},
                'trends': {'vector_count': 312},
                'xai': {'vector_

### Final cleanup

Release encoders and the Pinecone client. Clear torch's MPS/CUDA caches. After this cell the kernel holds almost nothing — you can leave it open indefinitely without memory pressure.


In [13]:
# Drop heavy objects so this kernel is cheap to keep open.
for name in ("dense_encoder", "bm25_encoder", "index", "pc"):
    try:
        del globals()[name]
    except KeyError:
        pass

# Clear torch caches if applicable
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()
except Exception:
    pass

mem("after final cleanup")
print("\n✓ Notebook A complete. Open Notebook B to run queries.")


  [MEM] after final cleanup              RSS =   1758.0 MB

✓ Notebook A complete. Open Notebook B to run queries.


### Notes

- **When to re-run this notebook:** when transaction CSVs change, amenities are updated, or the XAI bundle changes. Otherwise, the Pinecone index and BM25 cache are reused indefinitely by Notebook B.
- **To rebuild from the full 1.57M corpus:** set `SAMPLE_SIZE = None` in the config cell and re-run. Watch the `[MEM]` checkpoints — if RSS climbs above ~10 GB on a 16 GB Mac, lower `UPSERT_BATCH_SIZE` from 64 to 32 or 16.
- **To force a BM25 refit:** delete `bm25_encoder_v3.pkl` before running.
